In [25]:
%pip install pdfplumber
%pip install playwright
import json 
import pdfplumber
import os
import re
import json
import asyncio
import re
from playwright.async_api import async_playwright
from bs4 import BeautifulSoup

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Users\rlee0\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Users\rlee0\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [19]:
# --- CONFIGURATION ---
FILE_PATH = r"C:\Users\rlee0\OneDrive - The Ohio State University\AppsOSU\pathway\Pathway-AI\backend\files\Curriculum Sheet_BSBA_Finance.pdf"
OUTPUT_PATH = r"C:\Users\rlee0\OneDrive - The Ohio State University\AppsOSU\pathway\Pathway-AI\backend\curriculum_data.json"

# Regex pattern to identify real course codes (e.g., BUSFIN 4211, MATH 1151, ECON 2001.01)
COURSE_REGEX = r'[A-Z]{2,10}\s+\d{4}[A-Z]*(\.\d+)?'

def extract_course_ids(text):
    """Finds all course codes within a block of text and returns them as a clean list."""
    if not text:
        return []
    # Standardize the text (remove newlines and extra spaces)
    clean_text = " ".join(text.split())
    return [match.group() for match in re.finditer(COURSE_REGEX, clean_text)]

def clean_row_data(row):
    """Cleans a single row of text and checks if it contains a course."""
    # Flatten the row into a single string to search for course codes
    row_text = " ".join([str(cell) for cell in row if cell])
    found_courses = extract_course_ids(row_text)
    
    # If no course code is found, this is likely a header or footer (Return None)
    if not found_courses:
        return None
    
    return {
        "id": found_courses[0], # The first course found is the Subject Course
        "full_text": row_text,
        "raw_cells": row
    }

def parse_curriculum(pdf_path):
    if not os.path.exists(pdf_path):
        return {"error": "File not found"}

    data = {
        "degree": "Bachelor of Science in Business Administration",
        "major": "Finance",
        "total_hours_required": 121,
        "curriculum": {
            "business_foundation_courses": [],
            "major_core": [],
            "finance_specialization": []
        }
    }

    with pdfplumber.open(pdf_path) as pdf:
        all_rows = []
        for page in pdf.pages:
            tables = page.extract_tables()
            for table in tables:
                all_rows.extend(table)

        for row in all_rows:
            processed = clean_row_data(row)
            if not processed:
                continue
            
            course_id = processed["id"]
            row_text = processed["full_text"]
            
            # --- Prerequisite Extraction ---
            # We look for all courses in the row EXCEPT the main ID
            all_found = extract_course_ids(row_text)
            # Prereqs are usually every course mentioned AFTER the first one in the table row
            prereqs = all_found[1:] if len(all_found) > 1 else []

            # --- Title & Hours Extraction ---
            # Try to grab hours (usually a number like 1.5, 3, or 5)
            hours_match = re.search(r'\b(\d(\.\d)?)\b', row_text)
            hours = hours_match.group(1) if hours_match else ""

            course_obj = {
                "id": course_id,
                "prereqs": prereqs,
                "hours": hours
            }

            # --- ROUTING LOGIC (Categorization) ---
            # Business Foundations (Math, Econ, Acct, etc.)
            if any(x in course_id for x in ["MATH", "STAT", "ECON", "ACCTMIS", "CSE", "ENGLISH"]):
                data["curriculum"]["business_foundation_courses"].append(course_obj)
            
            # Specialization (Finance 4000+ level)
            elif "BUSFIN" in course_id and any(char.isdigit() and int(char) >= 4 for char in course_id if char.isdigit()):
                data["curriculum"]["finance_specialization"].append(course_obj)
            
            # Major Core (The rest)
            else:
                data["curriculum"]["major_core"].append(course_obj)

    return data

# --- EXECUTION ---
if __name__ == "__main__":
    final_data = parse_curriculum(FILE_PATH)
    with open(OUTPUT_PATH, "w") as f:
        json.dump(final_data, f, indent=4)
    print(f"Clean JSON created at: {OUTPUT_PATH}")

Clean JSON created at: C:\Users\rlee0\OneDrive - The Ohio State University\AppsOSU\pathway\Pathway-AI\backend\curriculum_data.json


In [21]:
# --- CONFIGURATION ---
FILE_PATH = r"C:\Users\rlee0\OneDrive - The Ohio State University\AppsOSU\pathway\Pathway-AI\backend\files\purple_bs_cse_requirements_and_sample_schedule_au18_3341_revision_rev_071720_purple.pdf"
OUTPUT_PATH = r"C:\Users\rlee0\OneDrive - The Ohio State University\AppsOSU\pathway\Pathway-AI\backend\cse_curriculum_data.json"

# Detects courses like CSE 2221, MATH 1151, or ECE 2020
COURSE_REGEX = r'[A-Z]{2,10}\s+\d{4}[A-Z]*'

def extract_course_ids(text):
    if not text: return []
    return [match.group() for match in re.finditer(COURSE_REGEX, text.replace('\n', ' '))]

def parse_cse_curriculum(pdf_path):
    if not os.path.exists(pdf_path):
        return {"error": "File not found"}

    data = {
        "degree": "Bachelor of Science in Computer Science and Engineering",
        "major": "CSE",
        "curriculum": {
            "general_engineering": [],
            "cse_core": [],
            "math_science_electives": []
        }
    }

    with pdfplumber.open(pdf_path) as pdf:
        all_rows = []
        for page in pdf.pages:
            tables = page.extract_tables()
            for table in tables:
                all_rows.extend(table)

        for row in all_rows:
            # Join row text to find all courses and numbers
            row_text = " ".join([str(cell) for cell in row if cell])
            found_courses = extract_course_ids(row_text)
            
            if not found_courses:
                continue

            # Extract hours (look for numbers like 3, 4, or 10)
            hours_match = re.search(r'\b(\d{1,2})\b', row_text)
            total_hours = int(hours_match.group(1)) if hours_match else 0
            
            # If a row has 2 courses and 8 hours, split them to 4 each
            per_course_hours = total_hours / len(found_courses) if len(found_courses) > 0 else 0

            for course_id in found_courses:
                course_obj = {
                    "id": course_id,
                    "hours": per_course_hours,
                    "prereqs": [] # Note: CSE prereqs are often logic-based; see below
                }

                # --- ROUTING ---
                if "CSE" in course_id:
                    data["curriculum"]["cse_core"].append(course_obj)
                elif any(x in course_id for x in ["ENGR", "MATH", "PHYSICS"]):
                    data["curriculum"]["general_engineering"].append(course_obj)
                else:
                    data["curriculum"]["math_science_electives"].append(course_obj)

    return data

# --- EXECUTION ---
if __name__ == "__main__":
    final_data = parse_cse_curriculum(FILE_PATH)
    with open(OUTPUT_PATH, "w") as f:
        json.dump(final_data, f, indent=4)
    print(f"CSE JSON created at: {OUTPUT_PATH}")

CSE JSON created at: C:\Users\rlee0\OneDrive - The Ohio State University\AppsOSU\pathway\Pathway-AI\backend\cse_curriculum_data.json


In [33]:

# --- 0. ENVIRONMENT SETUP ---
# Apply the patch for Jupyter/Windows to allow nested event loops
nest_asyncio.apply()
if sys.platform == 'win32':
    asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())

# --- 1. EXTRACT UNIQUE COURSES FROM YOUR JSONs ---
def get_unique_courses():
    course_ids = set()
    # List of your curriculum JSON files
    files = ['curriculum_data.json', 'cse_curriculum_data.json']
    
    for file in files:
        try:
            with open(file, 'r') as f:
                data = json.load(f)
                # Dig through the 'curriculum' object
                for category in data['curriculum'].values():
                    # Check if category is a list (CSE) or a dict with 'required_courses' (Finance)
                    course_list = category if isinstance(category, list) else category.get('required_courses', [])
                    for course in course_list:
                        # Clean ID: removes '+' or extra text (e.g., 'ACCTMIS 2200+' -> 'ACCTMIS 2200')
                        clean_id_match = re.search(r'[A-Z]+\s+\d{4}', course['id'])
                        if clean_id_match:
                            course_ids.add(clean_id_match.group())
        except FileNotFoundError:
            print(f"⚠️ {file} not found. Skipping...")
            
    return sorted(list(course_ids))

# --- 2. THE SCRAPING ENGINE ---
async def scrape_osu_times(course_list):
    results = {}
    async with async_playwright() as p:
        # Launch browser (headless=True runs it in the background)
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page()

        for course in course_list:
            print(f"🔍 Searching for sections: {course}...")
            # Term 1264 = Autumn 2026 (OSU Term Coding)
            url = f"https://classes.osu.edu/#/?q={course.replace(' ', '%20')}&client=class-search-ui&campus=col&term=1264"
            
            try:
                await page.goto(url)
                # Wait for the search results to actually appear
                await page.wait_for_selector('.result-row', timeout=6000)
                
                content = await page.content()
                soup = BeautifulSoup(content, 'html.parser')
                
                sections = []
                for row in soup.select('.result-row'):
                    sections.append({
                        "section_num": row.select_one('.class-number').text.strip() if row.select_one('.class-number') else "N/A",
                        "time": row.select_one('.time').text.strip() if row.select_one('.time') else "TBA",
                        "days": row.select_one('.days').text.strip() if row.select_one('.days') else "TBA",
                        "instructor": row.select_one('.instructor').text.strip() if row.select_one('.instructor') else "Staff",
                        "location": row.select_one('.location').text.strip() if row.select_one('.location') else "TBD"
                    })
                results[course] = sections
            except:
                # If the page times out or no classes are found
                results[course] = []

            # Respectful delay to prevent being blocked
            await asyncio.sleep(0.5)

        await browser.close()
    return results

# --- 3. EXECUTION ---
async def main():
    courses = get_unique_courses()
    if not courses:
        print("❌ No courses found to scrape. Ensure your JSON files are in the same directory.")
        return

    print(f"✅ Found {len(courses)} unique courses across your curriculum files.")
    
    course_times_data = await scrape_osu_times(courses)
    
    # Save the live times to a new JSON
    output_path = 'course_times.json'
    with open(output_path, 'w') as f:
        json.dump(course_times_data, f, indent=4)
    
    print(f"🎉 Success! Course times saved to {output_path}")

# Run the main function directly for Jupyter compatibility
await main()

✅ Found 85 unique courses across your curriculum files.


NotImplementedError: 

In [ ]:
import json
import asyncio
import re
import sys
import nest_asyncio
from playwright.async_api import async_playwright
from bs4 import BeautifulSoup

# --- 0. THE CRITICAL WINDOWS FIX ---
# This must happen before any async calls
if sys.platform == 'win32':
    asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())

nest_asyncio.apply()

# --- 1. EXTRACT UNIQUE COURSES FROM YOUR JSONs ---
def get_unique_courses():
    course_ids = set()
    files = ['curriculum_data.json', 'cse_curriculum_data.json']
    
    for file in files:
        try:
            with open(file, 'r') as f:
                data = json.load(f)
                for category in data['curriculum'].values():
                    # Handle Finance (dict) vs CSE (list)
                    course_list = category if isinstance(category, list) else category.get('required_courses', [])
                    for course in course_list:
                        clean_id_match = re.search(r'[A-Z]+\s+\d{4}', course['id'])
                        if clean_id_match:
                            course_ids.add(clean_id_match.group())
        except FileNotFoundError:
            print(f"⚠️ {file} not found. Skipping...")
            
    return sorted(list(course_ids))

# --- 2. THE SCRAPING ENGINE ---
async def scrape_osu_times(course_list):
    results = {}
    async with async_playwright() as p:
        # Launch browser
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page()

        for course in course_list:
            print(f"🔍 Searching: {course}...")
            # Term 1264 = AU 2026
            url = f"https://classes.osu.edu/#/?q={course.replace(' ', '%20')}&client=class-search-ui&campus=col&term=1264"
            
            try:
                await page.goto(url)
                await page.wait_for_selector('.result-row', timeout=6000)
                
                content = await page.content()
                soup = BeautifulSoup(content, 'html.parser')
                
                sections = []
                for row in soup.select('.result-row'):
                    sections.append({
                        "section_num": row.select_one('.class-number').text.strip() if row.select_one('.class-number') else "N/A",
                        "time": row.select_one('.time').text.strip() if row.select_one('.time') else "TBA",
                        "days": row.select_one('.days').text.strip() if row.select_one('.days') else "TBA",
                        "instructor": row.select_one('.instructor').text.strip() if row.select_one('.instructor') else "Staff",
                        "location": row.select_one('.location').text.strip() if row.select_one('.location') else "TBD"
                    })
                results[course] = sections
            except:
                results[course] = [] # No sections found

            await asyncio.sleep(0.5)

        await browser.close()
    return results

# --- 3. EXECUTION ---
async def main():
    courses = get_unique_courses()
    if not courses:
        print("❌ No courses found in JSON files.")
        return

    print(f"✅ Ready to scrape {len(courses)} courses.")
    
    course_times_data = await scrape_osu_times(courses)
    
    with open('course_times.json', 'w') as f:
        json.dump(course_times_data, f, indent=4)
    
    print(f"🎉 Success! Check course_times.json")

# IMPORTANT: In Jupyter, just await the main function
await main()